# EmpowerLens — D20 multi-task model (Kaggle GPU)

Trains the **D20 multi-task model** (`src/train_multitask.py`): one MentalRoBERTa encoder with three heads —
binary (weighted CE), multilabel over the 10 distortions (BCE), and sentence selection (mean-pooled token
states per sentence, BCE). The selection loss is **masked to 0** for No-Distortion rows and the 13 rows whose
quote could not be located (test B-06). Seeds **42, 1337, 2024**, D12 recipe: lr 3e-5, 5 epochs, weight decay 0.1,
max length 512, batch 16, loss weights 1/1/1.

All choices are made on **val**: the epoch (mean of val binary macro-F1, multilabel macro-F1, selector F1), the
binary threshold (lowest with val FPR ≤ 0.15), per-type multilabel thresholds, and the selector threshold.
`test.csv` is read once per seed, after training, and scored with those val choices.
About 25–35 min per seed on a T4.

**Before running:** Accelerator **GPU (T4 or P100)**, **Internet: On**, Kaggle secret **`HF_TOKEN`**
(`mental/mental-roberta-base` is gated).

**Resumable:** a seed is skipped if its `eval_multitask_<seed>.json` is already in `/kaggle/working/d20/`.
Checkpoints live in `/kaggle/temp` and are deleted after each seed (weights are never kept).

**After it finishes:** download `d20_multitask.zip`, copy the three `eval_multitask_<seed>.json` (and, optionally,
`run_<seed>_meta.json` / `epoch_history_<seed>.csv`) into `results/results_multitask/`,
then run `venv\Scripts\python.exe -m src.multitask_report` locally to rebuild `summary.md`.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls src/train_multitask.py src/multitask_report.py results/results_sentence_label/sentence_labels_train.csv results/results_sentence_label/excluded_rows.csv && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEEDS  = [42, 1337, 2024]
OUT    = Path("/kaggle/working/d20")
CK     = Path("/kaggle/temp/ckpt_d20")
OUT.mkdir(parents=True, exist_ok=True)

def recipe(seed):
    return (f"--model {MODEL} --seed {seed} --lr 3e-5 --epochs 5 --weight-decay 0.1 "
            f"--max-length 512 --batch-size 16 --w-binary 1.0 --w-multilabel 1.0 --w-selection 1.0 "
            f"--target-fpr 0.15 --splits data/splits")

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def load(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

print("config ok:", MODEL, SEEDS, OUT)

In [ ]:
# 3. Smoke test (~3 min): 100 rows per split, 1 epoch, max length 128. Writes nothing to OUT.
sh(f"python -m src.train_multitask --model {MODEL} --seed 42 --smoke --max-length 128 --out /kaggle/temp/smoke")
smoke_ck = next(Path("/kaggle/temp/smoke").glob("multitask_*_smoke"))   # smoke runs save as <name>_smoke
ev = load(smoke_ck / "eval_multitask_42_smoke.json")
assert (smoke_ck / "meta.json").exists() and (smoke_ck / "model.pt").exists()
print({p: {"spec": round(ev["splits"][p]["binary"]["specificity"], 3),
           "ml_macro_f1": round(ev["splits"][p]["multilabel"]["macro_f1"], 3),
           "sel_f1": round(ev["splits"][p]["selector"]["f1"], 3)} for p in ("val", "test")})
print("SMOKE OK")
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True)

In [ ]:
# 4. Per seed: train the multi-task model (D12 recipe + failed-run rule on VAL), keep the eval file, drop the weights.
log = {}
for seed in SEEDS:
    ev_path = OUT / f"eval_multitask_{seed}.json"
    if ev_path.exists():
        print(f"seed {seed}: already done, skipping"); continue
    ck = CK / f"multitask_{TAG}_{seed}"
    shutil.rmtree(CK, ignore_errors=True)
    sh(f"python -m src.train_multitask {recipe(seed)} --out {CK} --results-dir {OUT}")
    f1, reruns = load(ev_path)["splits"]["val"]["binary"]["macro_f1_argmax"], 0
    if f1 < 0.5:   # D12 failed-run rule (decided on val only): re-run once with identical settings
        print(f"[failed-run rule] seed {seed} val binary macro-F1 {f1:.3f} < 0.5 -> re-running once")
        shutil.rmtree(CK, ignore_errors=True)
        sh(f"python -m src.train_multitask {recipe(seed)} --out {CK} --results-dir {OUT}")
        f1, reruns = load(ev_path)["splits"]["val"]["binary"]["macro_f1_argmax"], 1
    shutil.copy(ck / "meta.json", OUT / f"run_{seed}_meta.json")
    shutil.copy(ck / "epoch_history.csv", OUT / f"epoch_history_{seed}.csv")
    ev = load(ev_path)
    log[seed] = {"val_binary_macro_f1": f1, "failed_run_reruns": reruns,
                 "best_epoch": ev["settings"]["best_epoch"], "binary_threshold": ev["thresholds"]["binary"]}
    (OUT / f"run_{seed}.json").write_text(json.dumps(log[seed], indent=2), encoding="utf-8")
    shutil.rmtree(CK, ignore_errors=True)   # weights are never kept
print(json.dumps(log, indent=2))

In [ ]:
# 5. Build summary.md (multi-task vs flat vs cascade) + zip for download.
Path("results/results_multitask").mkdir(parents=True, exist_ok=True)
for seed in SEEDS:
    shutil.copy(OUT / f"eval_multitask_{seed}.json", f"results/results_multitask/eval_multitask_{seed}.json")
sh("python -m src.multitask_report")
shutil.copy("results/results_multitask/summary.md", OUT / "summary.md")
!cd /kaggle/working && rm -f d20_multitask.zip && zip -qr d20_multitask.zip d20
!ls -la /kaggle/working/d20